# 🌍 Global Wind Patterns in the Atmosphere
### IEEE SciVis 2026 Contest — Interactive Atmospheric Dashboard

[![Python](https://img.shields.io/badge/Python-3.9%2B-blue)](https://python.org) [![Jupyter](https://img.shields.io/badge/Jupyter-Notebook-orange)](https://jupyter.org) [![License: MIT](https://img.shields.io/badge/License-MIT-green)](LICENSE)

---

**Dataset:** DYAMOND C1440-LLC2160 (GEOS atmospheric model + MITgcm ocean model)  
**Source:** [NASA NSDF — National Research Data Fabric](https://nationalresearchplatform.org)  
**Resolution:** ~7 km atmosphere / 2–4 km ocean  
**Simulation period:** 2020-01-20 → 2021-03-27 · Hourly cadence · 10,366 time steps  

---

## 📌 Table of Contents
1. [Installation](#1-installation)
2. [Imports & Configuration](#2-imports--configuration)
3. [Data Loading via OpenVisus](#3-data-loading-via-openvisus)
4. [Regions of Interest](#4-regions-of-interest)
5. [Helper Functions](#5-helper-functions)
6. [Dashboard Rendering](#6-dashboard-rendering)
7. [Interactive Widget Controls](#7-interactive-widget-controls)

---

## 1. Installation

Run this cell once to install all required dependencies. Restart the kernel after installation before proceeding.

In [ ]:
# Install required packages
!pip install -q cython cartopy OpenVisus ipywidgets scipy

# Verify key packages
import importlib
for pkg in ['numpy', 'matplotlib', 'cartopy', 'OpenVisus', 'ipywidgets', 'scipy']:
    try:
        importlib.import_module(pkg if pkg != 'OpenVisus' else 'OpenVisus')
        print(f'  ✓ {pkg}')
    except ImportError:
        print(f'  ✗ {pkg} — installation may have failed')

## 2. Imports & Configuration

Global imports, matplotlib style settings, and color palette used throughout the dashboard.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import OpenVisus as ov
from matplotlib.gridspec import GridSpec
from matplotlib.patches import FancyBboxPatch
from scipy.interpolate import griddata
from scipy.ndimage import zoom as ndzoom
from IPython.display import display, clear_output
from datetime import datetime, timedelta
import ipywidgets as widgets
from ipywidgets import Layout
import warnings
warnings.filterwarnings('ignore')

# ── Color palette (light scientific theme) ──────────────────────────────
BG         = '#EEF2F7'   # figure background
PANEL_BG   = '#FFFFFF'   # map panel background
TEXT_COLOR = '#1a1a2e'   # primary text
ACCENT     = '#1F4E79'   # titles and borders

# Apply globally so every figure inherits the theme
plt.rcParams.update({
    'figure.facecolor' : BG,
    'axes.facecolor'   : PANEL_BG,
    'font.family'      : 'DejaVu Sans',
    'text.color'       : TEXT_COLOR,
    'axes.labelcolor'  : TEXT_COLOR,
    'xtick.color'      : TEXT_COLOR,
    'ytick.color'      : TEXT_COLOR,
    'axes.edgecolor'   : '#aabbcc',
    'grid.color'       : '#aabbcc',
})

print('✓ Imports and configuration complete.')

## 3. Data Loading via OpenVisus

All variables are streamed live from the **NASA NSDF cloud** via OpenVisus — no local download required.  
Each variable is loaded with a graceful fallback: if a component is unavailable, the dashboard continues with zeros or synthetic data so the interface always renders.

| Variable | Description | Units |
|----------|-------------|-------|
| **U** | Eastward wind component | m/s |
| **V** | Northward wind component | m/s |
| **W** | Vertical velocity | Pa/s |
| **T** | Air temperature | K |
| **FCLD** | Cloud fraction | 0–1 |
| **H** | Geopotential height | m |
| **QI** | Cloud ice water mixing ratio | kg/kg |
| **QL** | Cloud liquid water mixing ratio | kg/kg |
| **DTHDT** | Convective temperature tendency | K/s |

In [ ]:
BASE_URL = (
    'https://nsdf-climate3-origin.nationalresearchplatform.org:50098'
    '/nasa/nsdf/climate3/dyamond/'
)

def _load(path, label):
    """Load an OpenVisus dataset; return (db, True) on success, (None, False) on failure."""
    try:
        db = ov.LoadDataset(BASE_URL + path)
        print(f'  ✓ {label}')
        return db, True
    except Exception as exc:
        print(f'  ⚠ {label} unavailable — {exc}')
        return None, False

print('Loading datasets from NSDF cloud...')
db_u,     _       = _load('mit_output/llc2160_arco/visus.idx',                          'U  (eastward wind)')
db_v,     _       = _load('mit_output/llc2160_v/v_llc2160_x_y_depth.idx',               'V  (northward wind)')
db_w,     USE_W   = _load('GEOS/GEOS_W/w_face_0_depth_52_time_0_10269.idx',             'W  (vertical velocity)')
db_t,     USE_T   = _load('GEOS/GEOS_T/t_face_0_depth_52_time_0_10269.idx',             'T  (temperature)')
db_fcld,  USE_FCLD= _load('GEOS/GEOS_FCLD/fcld_face_0_depth_52_time_0_10269.idx',       'FCLD (cloud fraction)')
db_h,     USE_H   = _load('GEOS/GEOS_H/h_face_0_depth_52_time_0_10269.idx',             'H  (geopotential height)')
db_qi,    USE_QI  = _load('GEOS/GEOS_QI/qi_face_0_depth_52_time_0_10269.idx',           'QI (cloud ice water)')
db_ql,    USE_QL  = _load('GEOS/GEOS_QL/ql_face_0_depth_52_time_0_10269.idx',           'QL (cloud liquid water)')
db_dthdt, USE_DTHDT = _load('GEOS/GEOS_DTHDT/dthdt_face_0_depth_52_time_0_10269.idx',  'DTHDT (convective tendency)')

# ── Pressure level → Z-index mapping ────────────────────────────────────
PRESSURE_LEVELS = {1000:0, 925:1, 850:2, 700:3, 600:4, 500:5,
                   400:6, 300:7, 250:8, 200:9, 150:10, 100:11}
PRESSURE_TO_Z   = {1000:51, 925:49, 850:46, 700:40, 600:35, 500:29,
                   400:22, 300:15, 250:11, 200:8,  150:5,  100:2}

# ── Time axis ────────────────────────────────────────────────────────────
SIM_START   = datetime(2020, 1, 20, 0, 0)
N_TIMESTEPS = 10_366          # verified via dataset metadata
DT_HOURS    = 1               # hourly cadence

TIME_LABELS = [
    (SIM_START + timedelta(hours=i)).strftime('%Y-%m-%d %H:%M UTC')
    for i in range(N_TIMESTEPS)
]

print(f'\n✓ Time axis ready.')
print(f'  First : {TIME_LABELS[0]}')
print(f'  Last  : {TIME_LABELS[-1]}')
print(f'  Steps : {N_TIMESTEPS:,}  '
      f'({N_TIMESTEPS/24:.0f} days = {N_TIMESTEPS/24/30.44:.1f} months, every {DT_HOURS} h)')

## 4. Regions of Interest

Twenty predefined geographic bounding boxes, from individual countries to ocean basins and polar regions.  
Each entry maps a display name to `(lon_min, lat_min, lon_max, lat_max)` in degrees.

In [ ]:
REGIONS = {
    '🌍 Global'          : (-180, -90,  180,  90),
    '🇲🇽 Mexico'          : (-120,  14,  -86,  33),
    '🇺🇸 United States'   : (-130,  24,  -65,  50),
    '🇨🇦 Canada'          : (-145,  42,  -52,  84),
    '🌎 North America'    : (-170,  15,  -50,  85),
    '🌎 South America'    : ( -82, -56,  -34,  13),
    '🇧🇷 Brazil'          : ( -74, -34,  -35,   5),
    '🌍 Europe'           : ( -25,  34,   45,  72),
    '🇷🇺 Russia'          : (  27,  41,  180,  82),
    '🌏 South/East Asia'  : (  60, -10,  150,  55),
    '🇨🇳 China'           : (  73,  18,  135,  54),
    '🇯🇵 Japan'           : ( 122,  24,  154,  46),
    '🇮🇳 India'           : (  67,   6,   98,  38),
    '🌍 Africa'           : ( -20, -38,   55,  38),
    '🌏 Oceania'          : ( 110, -50,  180, -10),
    '🌊 North Pacific'    : ( 140,  20, -120,  65),
    '🌊 North Atlantic'   : ( -80,  20,   20,  70),
    '🌊 Mediterranean'    : ( -10,  28,   42,  48),
    '❄ Arctic'           : (-180,  60,  180,  90),
    '❄ Antarctic'        : (-180, -90,  180, -60),
}

print(f'✓ {len(REGIONS)} regions defined.')

## 5. Helper Functions

Utility functions for data I/O, adaptive quality selection, array safety, and NaN interpolation.

In [ ]:
from functools import lru_cache
import threading

def read_geos(db, t, z_idx=29):
    """Read a GEOS variable at time step t and vertical index z_idx."""
    raw = np.array(db.read(time=t, quality=-6))
    if raw.ndim == 3:
        z_idx = min(z_idx, raw.shape[0] - 1)
        return raw[z_idx]
    return raw


def safe_levels(data, n=40, sym=False):
    """
    Compute n contour levels from 2nd-98th percentile.
    Reduced default from 80→40 levels for ~2x faster contourf rendering.
    """
    vmin = np.nanpercentile(data, 2)
    vmax = np.nanpercentile(data, 98)
    if sym:
        vmax = max(abs(vmin), abs(vmax))
        vmin = -vmax
    if vmax - vmin < 1e-10:
        vmin, vmax = vmin - 1, vmax + 1
    return np.linspace(vmin, vmax, n)


def ensure_min_size(arr, min_r=8, min_c=8):
    """Upsample arr if either spatial dimension is below minimum."""
    if arr is None or arr.size == 0:
        return np.zeros((min_r, min_c))
    r, c = arr.shape
    if r < min_r or c < min_c:
        zr = max(min_r / r, 1.0)
        zc = max(min_c / c, 1.0)
        arr = ndzoom(arr, (zr, zc), order=1)
    return arr


def pick_quality_and_step(region_name):
    """Adaptive quality/subsampling based on region bounding-box area."""
    lon0, lat0, lon1, lat1 = REGIONS.get(region_name, (-180, -90, 180, 90))
    area = abs(lon1 - lon0) * abs(lat1 - lat0)
    if   area > 50_000: return -18, 6
    elif area > 15_000: return -12, 4
    elif area >  4_000: return  -6, 2
    else:               return  -3, 1


def fill_nans(field):
    """
    Fast NaN fill: uses nearest-neighbour only (10x faster than griddata linear).
    Sufficient for visualization purposes.
    """
    if not np.any(np.isnan(field)):
        return field
    mask = np.isnan(field)
    # Use distance-transform nearest-neighbour fill
    from scipy.ndimage import distance_transform_edt
    idx = distance_transform_edt(mask, return_distances=False, return_indices=True)
    return field[tuple(idx)]


# ── LRU cache for data — avoids re-downloading same frame ─────────────────────
# Caches last 12 results (enough for play animation without re-fetching)
@lru_cache(maxsize=12)
def _cached_load(t, pressure, region):
    """Cached wrapper around the actual data fetch."""
    return _fetch_data(t, pressure, region)


print('✓ Helper functions defined (optimized).')


## 6. Dashboard Rendering

### 6a. `load_data` — Streaming & spatial cropping

Streams U/V from the LLC2160 ocean-atmosphere coupled grid and all GEOS atmospheric variables,  
then crops to the selected region's bounding box.

In [ ]:
def _fetch_data(t, pressure, region):
    """
    Internal: stream and crop data. Called only when cache misses.
    Do NOT call directly — use load_data() which handles the cache.
    """
    lon0, lat0, lon1, lat1 = REGIONS.get(region, (-180, -90, 180, 90))
    quality, step = pick_quality_and_step(region)

    try:
        u_full = np.array(db_u.read(time=t, z=[0, 1], quality=quality)[0])[::step, ::step]
        v_full = np.array(db_v.read(time=t, z=[0, 1], quality=quality)[0])[::step, ::step]
    except Exception as e:
        print(f'  [ERROR] Could not read U/V: {e}')
        return (np.zeros((8, 8)),) * 9 + ((lon0, lat0, lon1, lat1),)

    ny_f, nx_f = u_full.shape
    lon_full = np.linspace(-180, 180, nx_f)
    lat_full = np.linspace(-90,   90, ny_f)

    ix = np.where((lon_full >= lon0) & (lon_full <= lon1))[0]
    iy = np.where((lat_full >= lat0) & (lat_full <= lat1))[0]
    if len(ix) < 2 or len(iy) < 2:
        ix, iy = np.arange(nx_f), np.arange(ny_f)
    ix0, ix1 = ix[0], ix[-1] + 1
    iy0, iy1 = iy[0], iy[-1] + 1

    u   = ensure_min_size(u_full[iy0:iy1, ix0:ix1])
    v   = ensure_min_size(v_full[iy0:iy1, ix0:ix1])
    mag = np.sqrt(u**2 + v**2)
    ny, nx  = u.shape
    z_idx   = PRESSURE_TO_Z.get(pressure, 29)

    def geos_to_grid(db, use_flag):
        if not use_flag:
            return np.zeros((ny, nx))
        try:
            raw = read_geos(db, t, z_idx)
            if raw.shape != (ny_f, nx_f):
                raw = ndzoom(raw, (ny_f / raw.shape[0], nx_f / raw.shape[1]), order=1)
            return ensure_min_size(raw[iy0:iy1, ix0:ix1])
        except Exception as e:
            print(f'  [WARN] GEOS read error: {e}')
            return np.zeros((ny, nx))

    w      = geos_to_grid(db_w,     USE_W)
    t_fld  = geos_to_grid(db_t,     USE_T)
    fcld   = geos_to_grid(db_fcld,  USE_FCLD)
    h      = geos_to_grid(db_h,     USE_H)
    qi     = geos_to_grid(db_qi,    USE_QI)
    ql     = geos_to_grid(db_ql,    USE_QL)
    dthdt  = geos_to_grid(db_dthdt, USE_DTHDT)
    hydro  = qi + ql

    for fld in (t_fld, w, fcld, h, hydro, dthdt):
        fld[:] = fill_nans(fld)

    return u, v, mag, t_fld, fcld, h, w, hydro, dthdt, (lon0, lat0, lon1, lat1)


def load_data(t=0, pressure=500, region='🌍 Global'):
    """Public entry point — returns cached data when available."""
    return _cached_load(t, pressure, region)


print('✓ load_data defined (with LRU cache).')


### 6b. `create_dashboard` — Composite map rendering

Renders a two-panel figure: the main Plate Carrée map (left) with up to 7 toggleable layers,  
and a Key Insights reference panel (right).

**Layer rendering order (bottom → top):**
1. `T`    — Temperature background (RdYlBu_r colormap)
2. `W`    — Vertical velocity contours (RdBu_r, symmetric)
3. `FCLD` — Cloud fraction (Greys, semi-transparent)
4. `H`    — Geopotential height isohypses (gold contour lines)
5. `UV`   — Wind vector quivers (dark arrows)
6. `HYD`  — Hydrometeors QI+QL (GnBu, top-percentile mask)
7. `CONV` — Convective tendency DTHDT (YlOrRd / PuBu_r)

In [ ]:
def create_dashboard(t=0, pressure=500, layers=None, region='🌍 Global'):
    """
    Render the full interactive dashboard figure.
    Optimized: fewer contour levels, fast NaN fill, no canvas.draw() call.
    """
    if layers is None:
        layers = {'T': True, 'UV': True, 'W': True, 'FCLD': True,
                  'H': True, 'HYD': False, 'CONV': False}

    u, v, mag, t_field, fcld_f, h_f, w, hydro_f, dthdt_f, extent = \
        load_data(t, pressure, region)

    lon0, lat0, lon1, lat1 = extent
    ny, nx   = u.shape
    lon      = np.linspace(lon0, lon1, nx)
    lat      = np.linspace(lat0, lat1, ny)
    Lon, Lat = np.meshgrid(lon, lat)
    region_clean = region.split(' ', 1)[-1] if ' ' in region else region

    fig = plt.figure(figsize=(28, 12), facecolor=BG)
    fig.subplots_adjust(top=0.88, bottom=0.08, left=0.02, right=0.99, wspace=0.03)
    gs  = GridSpec(1, 2, width_ratios=[4.2, 1], figure=fig)
    ax1 = fig.add_subplot(gs[0], projection=ccrs.PlateCarree())
    ax2 = fig.add_subplot(gs[1])
    ax2.set_facecolor('#F0F5FB')
    ax2.axis('off')

    fig.text(0.5, 0.970, f'Wind Patterns in the Atmosphere  —  {region_clean}',
             ha='center', va='top', fontsize=26, fontweight='bold', color=TEXT_COLOR)
    fig.text(0.5, 0.945,
             f'DYAMOND GEOS  ·  Pressure Level: {pressure} hPa  ·  {TIME_LABELS[t]}',
             ha='center', va='top', fontsize=15, color=ACCENT, style='italic')
    fig.text(0.5, 0.922,
             'Data: DYAMOND C1440-LLC2160 — GEOS 7 km / MITgcm 2–4 km  |  '
             'Sim. start: 2020-01-20  |  Cadence: 1 h  |  '
             'Projection: Plate Carrée (EPSG:4326)',
             ha='center', va='top', fontsize=11, color='#555555')

    ax1.set_extent([lon0, lon1, lat0, lat1], crs=ccrs.PlateCarree())
    ax1.set_facecolor(PANEL_BG)
    ax1.coastlines(color='#333333', linewidth=1.6)
    ax1.add_feature(cfeature.BORDERS, linewidth=0.6, edgecolor='#666666')
    ax1.add_feature(cfeature.LAND,  facecolor='#e8ecf0', zorder=0)
    ax1.add_feature(cfeature.OCEAN, facecolor='#c8dff0', zorder=0)
    gl = ax1.gridlines(draw_labels=True, linewidth=0.4,
                       color='#aabbcc', alpha=0.8, linestyle='--')
    gl.top_labels = gl.right_labels = False
    gl.xlabel_style = gl.ylabel_style = {'color': TEXT_COLOR, 'fontsize': 11}

    im_T = im_W = im_FCLD = im_HYD = im_CONV = None

    # LAYER 1: Temperature (40 levels instead of 80)
    if layers.get('T'):
        try:
            valid_t = t_field[~np.isnan(t_field)]
            if valid_t.size:
                im_T = ax1.contourf(Lon, Lat, np.ma.masked_invalid(t_field),
                                    levels=safe_levels(valid_t, 40),
                                    cmap='RdYlBu_r', alpha=0.85, extend='both',
                                    transform=ccrs.PlateCarree())
        except Exception as e:
            print(f'  [WARN] T layer: {e}')

    # LAYER 2: Vertical velocity W (30 levels instead of 60)
    if layers.get('W'):
        try:
            valid_w = w[~np.isnan(w)]
            if valid_w.size:
                alpha_w = 0.6 if layers.get('T') else 0.9
                im_W = ax1.contourf(Lon, Lat, np.ma.masked_invalid(w),
                                    levels=safe_levels(valid_w, 30, sym=True),
                                    cmap='RdBu_r', alpha=alpha_w, extend='both',
                                    transform=ccrs.PlateCarree())
        except Exception as e:
            print(f'  [WARN] W layer: {e}')

    # LAYER 3: Cloud fraction FCLD (20 levels instead of 40)
    if layers.get('FCLD'):
        try:
            if USE_FCLD and np.nanmax(np.abs(fcld_f)) > 1e-6:
                fcld_d = np.clip(fcld_f, 0, 1)
            else:
                fcld_d = np.clip(
                    0.7 * np.exp(-((Lat) / 10)**2) +
                    0.4 * np.exp(-((np.abs(Lat) - 50) / 15)**2), 0, 1)
            valid_fc = fcld_d[~np.isnan(fcld_d)]
            if valid_fc.size:
                alpha_fc = 0.45 if layers.get('T') else 0.85
                im_FCLD = ax1.contourf(Lon, Lat, np.ma.masked_invalid(fcld_d),
                                       levels=safe_levels(valid_fc, 20),
                                       cmap='Greys', alpha=alpha_fc, extend='both',
                                       transform=ccrs.PlateCarree())
                if np.nanmax(fcld_d) > 0.4:
                    ax1.contour(Lon, Lat, np.ma.masked_invalid(fcld_d),
                                levels=[0.4], colors=['#1565C0'],
                                linewidths=0.8, alpha=0.7,
                                transform=ccrs.PlateCarree())
        except Exception as e:
            print(f'  [WARN] FCLD layer: {e}')

    # LAYER 4: Geopotential height H (8 contours instead of 14)
    if layers.get('H'):
        try:
            if np.nanmax(h_f) - np.nanmin(h_f) > 1e-6:
                cs_H = ax1.contour(Lon, Lat, np.ma.masked_invalid(h_f),
                                   levels=8, colors='#B8860B',
                                   linewidths=0.9, alpha=0.8,
                                   transform=ccrs.PlateCarree())
                ax1.clabel(cs_H, fmt='%d m', fontsize=8, colors='#B8860B')
        except Exception as e:
            print(f'  [WARN] H layer: {e}')

    # LAYER 5: Wind vectors U, V
    if layers.get('UV'):
        try:
            if not layers.get('T'):
                mag_clip = np.clip(mag, 0, np.nanpercentile(mag, 99))
                ax1.contourf(Lon, Lat, np.ma.masked_invalid(mag_clip * 80),
                             levels=40, cmap='turbo', alpha=0.85,
                             transform=ccrs.PlateCarree())
            step_q = max(1, min(nx, ny) // 30)
            ax1.quiver(Lon[::step_q, ::step_q], Lat[::step_q, ::step_q],
                         u[::step_q, ::step_q],   v[::step_q, ::step_q],
                       color='#1a1a2e', scale=80, width=0.0018, alpha=0.75,
                       transform=ccrs.PlateCarree())
        except Exception as e:
            print(f'  [WARN] UV layer: {e}')

    # LAYER 6: Hydrometeors
    if layers.get('HYD'):
        try:
            h_max = np.nanmax(hydro_f)
            if h_max > 1e-10:
                hydro_m = np.where(hydro_f > h_max * 0.05, hydro_f, np.nan)
                valid = hydro_m[~np.isnan(hydro_m)]
                if valid.size:
                    im_HYD = ax1.contourf(Lon, Lat, np.ma.masked_invalid(hydro_m),
                                          levels=safe_levels(valid, 20),
                                          cmap='GnBu', alpha=0.65, extend='max',
                                          transform=ccrs.PlateCarree())
                ax1.contour(Lon, Lat, np.ma.masked_invalid(hydro_f),
                            levels=[np.nanpercentile(hydro_f, 90)],
                            colors=['#0077B6'], linewidths=1.0, alpha=0.85,
                            transform=ccrs.PlateCarree())
        except Exception as e:
            print(f'  [WARN] HYD layer: {e}')

    # LAYER 7: Convective tendency
    if layers.get('CONV'):
        try:
            if np.nanmax(np.abs(dthdt_f)) > 1e-12:
                p85 = np.nanpercentile(dthdt_f, 85)
                p15 = np.nanpercentile(dthdt_f, 15)
                up  = np.where(dthdt_f >= p85, dthdt_f, np.nan)
                dn  = np.where(dthdt_f <= p15, dthdt_f, np.nan)
                if up[~np.isnan(up)].size:
                    im_CONV = ax1.contourf(Lon, Lat, np.ma.masked_invalid(up),
                                           levels=safe_levels(up[~np.isnan(up)], 12),
                                           cmap='YlOrRd', alpha=0.55, extend='max',
                                           transform=ccrs.PlateCarree())
                if dn[~np.isnan(dn)].size:
                    ax1.contourf(Lon, Lat, np.ma.masked_invalid(dn),
                                 levels=safe_levels(dn[~np.isnan(dn)], 12),
                                 cmap='PuBu_r', alpha=0.50, extend='min',
                                 transform=ccrs.PlateCarree())
                ax1.contour(Lon, Lat, np.ma.masked_invalid(dthdt_f),
                            levels=[p85], colors=['#D84315'],
                            linewidths=1.2, alpha=0.9,
                            transform=ccrs.PlateCarree())
        except Exception as e:
            print(f'  [WARN] CONV layer: {e}')

    active = [k for k, v in layers.items() if v]
    ax1.set_title(
        f'Composite Map  ·  Active layers: {" | ".join(active)}  ·  '
        f'{pressure} hPa  ·  {region_clean}',
        fontsize=15, pad=12, fontweight='bold', color=TEXT_COLOR)

    # Colorbar — fixed position, no canvas.draw() needed
    try:
        pos = ax1.get_position()
        cb_mappable, cb_label = None, ''
        if layers.get('T')    and im_T    is not None: cb_mappable, cb_label = im_T,    'Temperature (K)'
        elif layers.get('W')  and im_W    is not None: cb_mappable, cb_label = im_W,    'Vert. Velocity (Pa/s)'
        elif layers.get('FCLD') and im_FCLD is not None: cb_mappable, cb_label = im_FCLD,'Cloud Fraction'
        elif layers.get('HYD')  and im_HYD  is not None: cb_mappable, cb_label = im_HYD, 'Hydrometeors (kg/kg)'
        elif layers.get('CONV') and im_CONV  is not None: cb_mappable, cb_label = im_CONV,'Conv. Tendency (K/s)'
        if cb_mappable is not None:
            cax = fig.add_axes([pos.x1 + 0.003, pos.y0, 0.008, pos.height])
            cb  = fig.colorbar(cb_mappable, cax=cax, orientation='vertical')
            cb.set_label(cb_label, fontsize=11, color=TEXT_COLOR, labelpad=6)
            cb.ax.yaxis.set_tick_params(color=TEXT_COLOR, labelcolor=TEXT_COLOR, labelsize=9)
            cb.outline.set_edgecolor('#aabbcc')
    except Exception as e:
        print(f'  [WARN] Colorbar: {e}')

    ax1.text(0.01, 0.02,
             'Layers:  █ T (background)  — H isohypses (gold)  → U,V vectors  '
             '~ W contours (blue=descent / red=ascent)  · FCLD cloud boundary  '
             '◈ HYD hydrometeors (cyan)  ✦ CONV convection (orange=ascent / purple=subsidence)',
             transform=ax1.transAxes, fontsize=10, color='#1a1a2e',
             bbox=dict(boxstyle='round,pad=0.5', facecolor='#FFFFFF',
                       edgecolor='#2E75B6', alpha=0.92))

    insights = [
        ('① Jet Streams',    'Dominate mid-latitudes\n30–60°N/S'),
        ('② W Contours',     'Convective ascent (+)\nand subsidence (−)'),
        ('③ Temperature',    'Gradients drive\natmospheric circulation'),
        ('④ FCLD Boundary',  'Marks cloudy regions\ncorrelated with W > 0'),
        ('⑤ Isohypses (H)',  'Outline high/low\npressure systems'),
        ('⑥ HYD (QI+QL)',    'Precipitation-laden\nregions (ice + liquid)'),
        ('⑦ CONV (DTHDT)',   'Active deep convection\nzones (thunderstorms)'),
    ]

    ax2.set_xlim(0, 1); ax2.set_ylim(0, 1)
    ax2.add_patch(FancyBboxPatch(
        (0.03, 0.03), 0.94, 0.94, boxstyle='round,pad=0.02',
        linewidth=2, edgecolor=ACCENT, facecolor='#F0F5FB',
        transform=ax2.transAxes, zorder=0))
    ax2.text(0.5, 0.960, 'Key Insights', transform=ax2.transAxes,
             ha='center', va='top', fontsize=14, fontweight='bold', color=ACCENT)
    ax2.plot([0.06, 0.94], [0.932, 0.932], transform=ax2.transAxes,
             color=ACCENT, linewidth=2.0, alpha=0.5, solid_capstyle='round')

    y_start, y_end = 0.910, 0.055
    slot = (y_start - y_end) / len(insights)
    for i, (title, body) in enumerate(insights):
        y_top = y_start - i * slot
        if i % 2 == 0:
            ax2.add_patch(FancyBboxPatch(
                (0.04, y_top - slot + 0.003), 0.92, slot - 0.005,
                boxstyle='round,pad=0.005', linewidth=0, facecolor='#DDE8F5',
                alpha=0.55, transform=ax2.transAxes, zorder=1))
        ax2.text(0.08, y_top - 0.006, title, transform=ax2.transAxes,
                 ha='left', va='top', fontsize=10.5, fontweight='bold', color=ACCENT, zorder=2)
        ax2.text(0.08, y_top - 0.006 - slot * 0.40, body, transform=ax2.transAxes,
                 ha='left', va='top', fontsize=9.5, color='#2c3e50', linespacing=1.3, zorder=2)
        if i < len(insights) - 1:
            ax2.plot([0.06, 0.94], [y_top - slot + 0.005] * 2,
                     transform=ax2.transAxes, color='#B0C4DE', linewidth=0.8, alpha=0.8)

    plt.show()
    plt.close(fig)  # Free memory immediately after rendering


print('✓ create_dashboard defined (optimized).')


## 7. Interactive Widget Controls

Run this cell to launch the full interactive dashboard.  
All controls update the map in place without reloading the page.

| Control | Type | Description |
|---------|------|-------------|
| Region | Dropdown | 20 geographic regions |
| Pressure | Dropdown | 12 pressure levels (100–1000 hPa) |
| Date/Time | Dropdown | Exact UTC timestamp per step |
| Play / Slider | Animation | Step through the full 14-month simulation |
| Layer toggles | Checkboxes | 7 independent atmospheric layers |

In [ ]:
import time as _time

# ── Dropdowns ────────────────────────────────────────────────────────────
region_dropdown = widgets.Dropdown(
    options=list(REGIONS.keys()), value='🌍 Global',
    description='Region:', style={'description_width': 'initial'},
    layout=Layout(width='240px'))

pressure_dropdown = widgets.Dropdown(
    options=list(PRESSURE_LEVELS.keys()), value=500,
    description='Pressure (hPa):', style={'description_width': 'initial'},
    layout=Layout(width='200px'))

time_dropdown = widgets.Dropdown(
    options=[(label, i) for i, label in enumerate(TIME_LABELS)],
    value=0, description='Date/Time (UTC):',
    style={'description_width': 'initial'},
    layout=Layout(width='290px'))

# ── Playback — interval=1500ms so renders don't pile up ──────────────────
play_btn = widgets.Play(
    value=0, min=0, max=N_TIMESTEPS - 1, step=1,
    interval=1500,          # was 500 — gives render time to finish
    description='Play', disabled=False,
    layout=Layout(width='80px', height='36px'))

slider = widgets.IntSlider(
    min=0, max=N_TIMESTEPS - 1, step=1, value=0,
    readout=False, layout=Layout(width='400px'))

widgets.jslink((play_btn, 'value'), (slider, 'value'))

step_label = widgets.Label(value=f'Step 1 / {N_TIMESTEPS:,}  |  {TIME_LABELS[0]}')

# ── Layer toggles ─────────────────────────────────────────────────────────
layer_style = {'description_width': 'initial'}
cb_T    = widgets.Checkbox(value=True,  description='🌡 Temperature (T)',       style=layer_style, layout=Layout(width='190px'))
cb_UV   = widgets.Checkbox(value=True,  description='💨 Wind Vectors (U,V)',    style=layer_style, layout=Layout(width='190px'))
cb_W    = widgets.Checkbox(value=True,  description='⬆ Vert. Velocity (W)',     style=layer_style, layout=Layout(width='190px'))
cb_FCLD = widgets.Checkbox(value=True,  description='☁ Cloud Fraction',         style=layer_style, layout=Layout(width='175px'))
cb_H    = widgets.Checkbox(value=True,  description='〰 Geopotential (H)',      style=layer_style, layout=Layout(width='190px'))
cb_HYD  = widgets.Checkbox(value=False, description='🌧 Hydrometeors (QI+QL)',  style=layer_style, layout=Layout(width='210px'))
cb_CONV = widgets.Checkbox(value=False, description='⚡ Convection (DTHDT)',     style=layer_style, layout=Layout(width='200px'))

layer_box = widgets.HBox(
    [widgets.HTML("<b style='color:#1F4E79;font-size:13px'>🗂 Map Layers:</b>"),
     cb_T, cb_UV, cb_W, cb_FCLD, cb_H, cb_HYD, cb_CONV],
    layout=Layout(align_items='center', gap='6px', padding='6px 10px',
                  border='1px solid #2E75B6', border_radius='6px',
                  flex_flow='row wrap'))

info_html = widgets.HTML(
    value=(
        "<div style='background:#dbeafe;border:1px solid #2E75B6;"
        "border-radius:6px;padding:5px 12px;font-size:12px;color:#1e3a5f'>"
        f"<b>Dataset:</b> DYAMOND C1440-LLC2160 &nbsp;|&nbsp; "
        f"<b>Period:</b> 2020-01-20 → 2021-03-27 &nbsp;|&nbsp; "
        f"<b>Cadence:</b> 1 h &nbsp;|&nbsp; "
        f"<b>Total steps:</b> {N_TIMESTEPS:,} &nbsp;|&nbsp; "
        f"<b>Duration:</b> {N_TIMESTEPS / 24:.0f} days ≈ {N_TIMESTEPS / 24 / 30.44:.1f} months"
        "</div>"))

# ── Debounce + busy-lock state ────────────────────────────────────────────
output      = widgets.Output()
_updating   = False          # sync lock (prevents dropdown↔slider loop)
_busy       = False          # render lock (skip if already rendering)
_pending_t  = None           # last requested step while busy
_last_call  = 0.0            # timestamp of last call (for debounce)
DEBOUNCE_S  = 0.15           # 150 ms debounce for layer/region/pressure changes

def get_layers():
    return {'T': cb_T.value, 'UV': cb_UV.value, 'W': cb_W.value,
            'FCLD': cb_FCLD.value, 'H': cb_H.value,
            'HYD': cb_HYD.value, 'CONV': cb_CONV.value}

def _render(t):
    """Core render call — always runs in the output widget context."""
    global _busy, _pending_t
    _busy = True
    _pending_t = None
    try:
        with output:
            clear_output(wait=True)
            step_label.value = f'Step {t+1:,} / {N_TIMESTEPS:,}  |  {TIME_LABELS[t]}'
            create_dashboard(t=t, pressure=pressure_dropdown.value,
                             layers=get_layers(), region=region_dropdown.value)
    finally:
        _busy = False
        # If a new step arrived while we were rendering, draw it now
        if _pending_t is not None:
            _render(_pending_t)

def update_dashboard(change=None):
    """Debounced update — ignores rapid-fire calls within DEBOUNCE_S seconds."""
    global _last_call
    now = _time.time()
    if now - _last_call < DEBOUNCE_S:
        return
    _last_call = now
    _render(slider.value)

def update_dashboard_slider(change=None):
    """Slider update — skips render if busy, queues last step instead."""
    global _pending_t
    t = slider.value
    if _busy:
        _pending_t = t   # remember latest requested step
        return           # drop this frame — render will pick up _pending_t
    _render(t)

def on_slider_change(change):
    global _updating
    if _updating: return
    _updating = True
    time_dropdown.value = slider.value
    _updating = False
    update_dashboard_slider()

def on_time_dropdown_change(change):
    global _updating
    if _updating: return
    _updating = True
    slider.value = time_dropdown.value
    _updating = False
    update_dashboard_slider()

slider.observe(on_slider_change,               names='value')
time_dropdown.observe(on_time_dropdown_change, names='value')
pressure_dropdown.observe(update_dashboard,    names='value')
region_dropdown.observe(update_dashboard,      names='value')
for cb in [cb_T, cb_UV, cb_W, cb_FCLD, cb_H, cb_HYD, cb_CONV]:
    cb.observe(update_dashboard, names='value')

control_row = widgets.HBox(
    [region_dropdown, pressure_dropdown, time_dropdown,
     widgets.HTML("<b style='color:#1F4E79;font-size:13px'>▶ Animate:</b>"),
     play_btn, slider, step_label],
    layout=Layout(align_items='center', gap='10px', flex_flow='row wrap'))

display(info_html, layer_box, control_row, output)
update_dashboard()
